In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [3]:
df_features = pd.read_csv("../data/processed/task3_encoded_scaled.csv")
df_features.head()

,BMI_TCR,FUNC_STAT_TCR,INIT_CPRA,INIT_AGE,DIALYSIS_DATE,INIT_DATE,ON_DIALYSIS_Y,GENDER_M,ABO_A1,ABO_A1B,...,REGION_2,REGION_3,REGION_4,REGION_5,REGION_6,REGION_7,REGION_8,REGION_9,REGION_10,REGION_11
0,0.483082,2080.0,-0.32562,0.077090,2018-03-30,2020-03-25,True,False,False,False,...,False,False,False,False,False,False,True,False,False,False
1,0.209378,2070.0,-0.32562,0.282551,2017-08-16,2020-02-14,True,True,False,False,...,False,False,False,True,False,False,False,False,False,False
2,0.693094,2070.0,-0.32562,-0.333834,Not on dialysis,2020-05-27,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3,-1.518920,2090.0,-0.32562,0.624987,2019-01-05,2020-04-02,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False
4,-0.950854,2070.0,-0.32562,0.624987,2019-01-10,2020-02-05,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False


In [4]:
# parse dates
df_features['DIALYSIS_DATE_parsed'] = pd.to_datetime(df_features['DIALYSIS_DATE'], errors='coerce')
df_features['INIT_DATE'] = pd.to_datetime(df_features['INIT_DATE'], errors='coerce')

# get the duration on dialysis as of listing date, in days
df_features['DIALYSIS_DURATION_DAYS'] = (df_features['INIT_DATE'] - df_features['DIALYSIS_DATE_parsed']).dt.days

# patients never on dialysis (on_dialysis == 'N'/ False) -> duration = 0
df_features.loc[df_features['ON_DIALYSIS_Y'] == False, 'DIALYSIS_DURATION_DAYS'] = 0

# negative durations (dialysis started after listing)
# some patients have this, so we will keep it rather than clipping
print(df_features['DIALYSIS_DURATION_DAYS'].describe())
print('Negative durations:', (df_features['DIALYSIS_DURATION_DAYS'] < 0).sum())

# drop the now-redundant date columns
df_model = df_features.drop(columns=['DIALYSIS_DATE', 'DIALYSIS_DATE_parsed', 'INIT_DATE'])


count    494803.000000
mean        549.944622
std         951.196943
min       -3792.000000
25%           0.000000
50%         247.000000
75%         762.000000
max       15356.000000
Name: DIALYSIS_DURATION_DAYS, dtype: float64
Negative durations: 42372


In [5]:
print(df_features['DIALYSIS_DURATION_DAYS'].isna().sum())

0


In [8]:
# add to numerical columns to be scaled
new_numerical_col = ['DIALYSIS_DURATION_DAYS']

In [9]:
scaler = StandardScaler()
df_model[new_numerical_col] = scaler.fit_transform(df_model[new_numerical_col])
df_model[new_numerical_col].describe()

,DIALYSIS_DURATION_DAYS
count,4.948030e+05
mean,3.561308e-18
std,1.000001e+00
min,-4.564721e+00
25%,-5.781612e-01
50%,-3.184881e-01
75%,2.229355e-01
max,1.556573e+01
